# Chapter 10 — Trajectory Evaluation and Metrics

*Evaluate the path, not just the answer.*

## Objective

**Why.** Chatbot evaluation asks one question — was the answer good? Agent evaluation asks ~nine: right tools, right stop, right escalation, gates fired, within budget, claims supported, constraints respected, replayable.

**What.** Each question is about the *whole run*, not the final answer. That run is the **trajectory**, and every property worth measuring is a pure function over it — so the audit log can re-run the checks without rerunning the agent.

**How.** Collect a `Trajectory` from each of three contrasting runs (success, escalation, budget failure), apply the metric catalog, then do a claim-by-claim groundedness check.

In [ ]:
from pydantic import BaseModel
from forgeloop.core import (
    AgentState, BaseAgent, Budget, BudgetTracker, Escalate, Finish, TaskSpec, ToolCall, run_loop,
)
from proofloop.evaluation import (
    Claim, check_groundedness, collect, coverage, escalated, extract_claims,
    failed, finished_cleanly, groundedness_report, step_count, summarize,
    task_success, tool_call_count, tool_failure_count,
)
from forgeloop.tools import GovernedToolExecutor, RiskLevel, Tool, ToolRegistry

## Run three agents that end three different ways

In [ ]:
class PingNTimes(BaseAgent):
    def __init__(self, n): self.n = n
    def propose_action(self, state):
        if state.step >= self.n: return Finish(output='done')
        return ToolCall(tool_name='ping', arguments={})

class AlwaysEscalate(BaseAgent):
    def propose_action(self, state):
        return Escalate(reason='needs human')

# A real environment: every action runs through the Chapter 6 executor and a
# real (if trivial) tool, not a canned {'success': True}.
class PingIn(BaseModel): pass
class PingOut(BaseModel): ok: bool

registry = ToolRegistry()
registry.register(Tool(name='ping', description='a no-op tool',
                       input_schema=PingIn, output_schema=PingOut,
                       risk=RiskLevel.LOW, fn=lambda: {'ok': True}))

class ToolEnv:
    def __init__(self, executor): self._ex = executor
    def step(self, action):
        r = self._ex.execute(action)
        return {'success': r.success, 'output': r.output, 'error': r.error}

env = ToolEnv(GovernedToolExecutor(registry))
task = TaskSpec(goal='trajectory demo')

happy   = collect(task, run_loop(PingNTimes(3), env, AgentState(task=task), max_steps=10))
esc     = collect(task, run_loop(AlwaysEscalate(), env, AgentState(task=task), max_steps=10))
starved = collect(task, run_loop(PingNTimes(10), env, AgentState(task=task), max_steps=10,
                                 budget_tracker=BudgetTracker(Budget(tool_calls=2))))

## Summaries are pure functions over a Trajectory

In [ ]:
for name, traj in [('happy', happy), ('escalated', esc), ('budget-failed', starved)]:
    s = summarize(traj)
    print(f'{name:>14}: status={s["status"]:<10} steps={s["steps"]:<2} '
          f'tool_calls={s["tool_calls"]:<2} escalated={s["escalated"]} failed={s["failed"]}')

## Individual metrics compose

Each metric is independent. You can build your own dashboard by composing them. Tool failures, escalation rate, finished-cleanly rate are the high-leverage ones.

In [ ]:
for name, traj in [('happy', happy), ('escalated', esc), ('budget-failed', starved)]:
    print(f'{name:>14}: success={task_success(traj):.1f} clean={finished_cleanly(traj)} '
          f'tool_failures={tool_failure_count(traj)}')

## Groundedness: extract claims, match evidence

The groundedness checker is deliberately naive (term overlap). Real systems use NLI or a verifier LM. The pattern matters more than the implementation: a claim that has no evidence is not a fact.

In [ ]:
answer = (
    'The overdraft fee is thirty-five dollars per occurrence. '
    'Customers are notified within one business day. '
    'Dinosaurs were vegetarian.'
)
evidence = {
    'overdraft': 'Overdraft fees apply at thirty-five dollars per occurrence.',
    'notice':    'Customers are notified within one business day of the posted overdraft.',
}

claims = extract_claims(answer)
report = groundedness_report(claims, evidence)
for r in report:
    print(f'  [{r.verdict.value:<11}] {r.claim!r}  evidence={r.evidence_id}')
print(f'\ncoverage: {coverage(report) * 100:.0f}%')

## An optional verifier: groundedness as distance

Strengthening the term-overlap check by reaching for a second LLM (*LLM-as-judge*) means a model judging a model — its verdict can't be replayed. A deterministic alternative: reduce a claim to a triple and ask how far it sits from a trained store. The distance is a continuous **groundedness score** — small for grounded, large for fabricated — and the same input always yields the same number. (The full claim-extracting judge is the Chapter 17 testing harness; here we use the `score_triple` primitive directly.)

In [ ]:
import torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore
from forgeloop.gms_backend import GMSMemory

_root = Path('.') if Path('data/gms_banking_store').exists() else Path('..')
store = GMSExpertStore(
    DocGMSConfig(store_path=str(_root / 'data' / 'gms_banking_store')),
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
)
store.load()
mem = GMSMemory(store)

# A claim reduced to a triple, scored against the store: distance = groundedness.
for claim, (h, r, t) in [('overdraft fee is $35', ('overdraft', 'has_fee_amount', '35.0')),
                         ('overdraft fee is $50', ('overdraft', 'has_fee_amount', '50.0'))]:
    print(f'  {claim:<22} groundedness={mem.score_triple(h, r, t):.3f}')

## Anti-patterns flagged here

- Evaluating only the final answer.
- Treating LLM-judge scores as ground truth without calibration.
- Reporting averages without distributions.

In [ ]:
# Self-check
assert task_success(happy) == 1.0
assert escalated(esc)
assert failed(starved)
assert coverage(report) > 0 and coverage(report) < 1
print('OK')

## Exercises

Worked solutions follow. They turn the metric catalog and groundedness check into the kind of report a release process gates on. Exercise 10.3 reuses the GMS store loaded above and loads Qwen2.5-3B-Instruct (~30s).

In [ ]:
# Exercise 10.1 — a suite metric is a function over many trajectories
def report(trajs):
    n = len(trajs)
    return {
        'success_rate':    sum(task_success(t) for t in trajs) / n,
        'escalation_rate': sum(escalated(t) for t in trajs) / n,
        'clean_rate':      sum(finished_cleanly(t) for t in trajs) / n,
    }

print(report([happy, esc, starved]))

In [ ]:
# Exercise 10.2 — a groundedness ship-gate
def ship_or_refuse(answer, evidence, threshold=1.0):
    rep = groundedness_report(extract_claims(answer), evidence)
    cov = coverage(rep)
    if cov >= threshold:
        return {'shipped': True, 'coverage': cov}
    unsupported = [r.claim for r in rep if r.verdict.value != 'supported']
    return {'shipped': False, 'coverage': cov, 'unsupported': unsupported}

print('dinosaur answer:', ship_or_refuse(answer, evidence))
fully_supported = ('The overdraft fee is thirty-five dollars per occurrence. '
                   'Customers are notified within one business day.')
print('clean answer:   ', ship_or_refuse(fully_supported, evidence))

In [ ]:
# Exercise 10.3 — two verifiers on the same claims
from forgeloop.models import QwenAdapter

judge_lm = QwenAdapter(
    system=("You are a strict fact-checker. Given a POLICY and a CLAIM, answer with exactly "
            "'yes' if the claim is supported by the policy, or 'no' if it is not."),
    max_new_tokens=4,
)
def llm_judge(claim, policy):
    return judge_lm.complete(f'POLICY: {policy}\nCLAIM: {claim}\nSupported?').strip().lower()

TAU = 1.45
policy = 'The overdraft fee is $35 per occurrence.'
for claim, (h, r, t) in [('overdraft fee is $35', ('overdraft', 'has_fee_amount', '35.0')),
                         ('overdraft fee is $50', ('overdraft', 'has_fee_amount', '50.0'))]:
    score = mem.score_triple(h, r, t)
    gms = 'grounded' if score <= TAU else 'fabricated'
    print(f'  {claim:<22} GMS={score:.3f} ({gms:<10}) LLM-judge={llm_judge(claim, policy)}')
# The GMS score replays byte-for-byte from the frozen store; the LLM verdict does not.